# P2-3 · CNN hiện đại và transfer learning

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torchvision
num_classes = 5
WEIGHTS = None     # thực tế: "IMAGENET1K_V1" (tải ~45 MB trọng số pretrain); trang này không tải được nên dùng None để chạy thử
m = torchvision.models.resnet18(weights=WEIGHTS)
m.fc = torch.nn.Linear(m.fc.in_features, num_classes)        # head mới: 512 → num_classes
count = lambda: sum(p.numel() for p in m.parameters() if p.requires_grad)

# Giai đoạn 1: linear probe
for p in m.parameters(): p.requires_grad = False
for p in m.fc.parameters(): p.requires_grad = True
opt = torch.optim.AdamW(m.fc.parameters(), lr=1e-3)
print("giai đoạn 1, tham số được train:", count())          # chỉ head: 512 × 5 + 5
# ... train vài epoch ...

# Giai đoạn 2: fine-tune toàn bộ, backbone lr nhỏ hơn
for p in m.parameters(): p.requires_grad = True
opt = torch.optim.AdamW([
    {"params": m.fc.parameters(), "lr": 1e-3},
    {"params": [p for n, p in m.named_parameters() if not n.startswith("fc")], "lr": 1e-4},
], weight_decay=0.01)
print("giai đoạn 2, tham số được train:", count())

giai đoạn 1, tham số được train: 2565
giai đoạn 2, tham số được train: 11179077


In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()      # resize, crop, chuẩn hóa đúng như lúc pretrain
preprocess

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [ ]:
import torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"                            # fp16 chỉ có lợi trên GPU
model = nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
train_loader = [(torch.randn(64, 3, 32, 32), torch.randint(0, 10, (64,))) for _ in range(5)]   # dữ liệu giả

scaler = torch.amp.GradScaler(device, enabled=use_amp)      # chỉ cần cho fp16
for x, y in train_loader:
    x, y = x.to(device), y.to(device)
    opt.zero_grad()
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        loss = F.cross_entropy(model(x), y)
    scaler.scale(loss).backward()                     # nhân loss lên để gradient nhỏ không thành 0
    scaler.step(opt)                                  # chia lại trước khi cập nhật, bỏ bước nếu có inf
    scaler.update()
print("thiết bị:", device, "| fp16:", use_amp, "| loss cuối:", round(loss.item(), 3))

thiết bị: cpu | fp16: False | loss cuối: 2.476


## Bài tập 4: Debug một vòng train hỏng {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
X = torch.randn(1024, 20); Y = (X @ torch.randn(20, 10)).argmax(1)       # 10 lớp, có quy luật tuyến tính
loader = [(X[i:i + 64], Y[i:i + 64]) for i in range(0, 1024, 64)]
criterion = nn.CrossEntropyLoss()
def make():
    torch.manual_seed(1)
    m = nn.Sequential(nn.Linear(20, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 10))
    return m, torch.optim.SGD(m.parameters(), lr=0.05)

In [ ]:
model, optimizer = make()
for epoch in range(5):
    model.eval()
    for x, y in loader:
        logits = model(x)
        loss = criterion(torch.softmax(logits, dim=1), y)
        loss.backward()
        optimizer.step()
    print("epoch", epoch, "loss", round(loss.item(), 3))

epoch 0 loss 2.299
epoch 1 loss 2.265
epoch 2 loss 2.22
epoch 3 loss 2.213
epoch 4 loss 2.132


In [ ]:
model, optimizer = make()
for epoch in range(5):
    model.train()
    for x, y in loader:
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    print("epoch", epoch, "loss", round(loss.item(), 3))

epoch 0 loss 2.21
epoch 1 loss 2.003
epoch 2 loss 1.849
epoch 3 loss 1.723
epoch 4 loss 1.58


## Bài tập 5: Đo gradient biến mất trong mạng sâu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn

class Deep(nn.Module):
    def __init__(self, act, residual, depth=20, width=32):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(width, width) for _ in range(depth)])
        self.act, self.residual = act, residual
    def forward(self, x):
        for l in self.layers:
            h = self.act(l(x))
            x = x + h if self.residual else h
        return x

def ti_le_grad(model):
    torch.manual_seed(1)
    x = torch.randn(64, 32)
    model(x).pow(2).mean().backward()
    g_dau = model.layers[0].weight.grad.norm().item()
    g_cuoi = model.layers[-1].weight.grad.norm().item()
    return g_dau / g_cuoi
# TODO: tl (dict tên → tỉ lệ gradient lớp đầu / lớp cuối) cho 3 mạng, mỗi mạng tạo sau torch.manual_seed(0)

In [ ]:
import math
tl = {}
for ten, act, res in [("sigmoid", torch.sigmoid, False), ("relu", torch.relu, False), ("relu + tắt", torch.relu, True)]:
    torch.manual_seed(0)
    tl[ten] = ti_le_grad(Deep(act, res))
    print(f"{ten:11s}: gradient lớp đầu / lớp cuối = {tl[ten]:.2e}")

sigmoid    : gradient lớp đầu / lớp cuối = 1.51e-17
relu       : gradient lớp đầu / lớp cuối = 1.12e-08
relu + tắt : gradient lớp đầu / lớp cuối = 2.73e-01


## Bài tập 6: Chuẩn bị ResNet-18 cho bài phân loại 5 loại sản phẩm {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from torchvision.models import resnet18
torch.manual_seed(0)
model = resnet18(weights=None)        # thực tế: weights=ResNet18_Weights.DEFAULT (cần tải ~45 MB); cơ chế giống hệt
x = torch.randn(4, 3, 224, 224); y = torch.tensor([0, 1, 2, 3])
def dem_train(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
# TODO: thay model.fc; giai đoạn 1: đóng băng mọi thứ trừ fc (n_gd1); một bước train, kiểm tra conv1 không đổi;
#       giai đoạn 2: mở layer4, optimizer có 2 nhóm lr (n_gd2)

In [ ]:
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 5)
n_gd1 = dem_train(model)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
truoc = model.conv1.weight.clone()
model.train()
opt.zero_grad(); nn.functional.cross_entropy(model(x), y).backward(); opt.step()
dung_yen = torch.equal(truoc, model.conv1.weight)

for p in model.layer4.parameters():
    p.requires_grad = True
n_gd2 = dem_train(model)
opt2 = torch.optim.AdamW([{"params": model.layer4.parameters(), "lr": 1e-4},
                          {"params": model.fc.parameters(), "lr": 1e-3}], weight_decay=0.01)
print(f"giai đoạn 1: train {n_gd1:,} tham số ({n_gd1 / sum(p.numel() for p in model.parameters()):.3%})")
print("conv1 đứng yên sau một bước:", dung_yen)
print(f"giai đoạn 2: train {n_gd2:,} tham số, lr: {[g['lr'] for g in opt2.param_groups]}")

giai đoạn 1: train 2,565 tham số (0.023%)
conv1 đứng yên sau một bước: True
giai đoạn 2: train 8,396,293 tham số, lr: [0.0001, 0.001]


## Bài tập 7: Ba định dạng số thực trong thực tế {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch
torch.manual_seed(0)
A, B = torch.randn(256, 256), torch.randn(256, 256)
# TODO: max16, g16, gbf, mot_bf, sai_so

In [ ]:
max16 = torch.finfo(torch.float16).max
print("float16 lớn nhất:", max16, "| 70000 thành", torch.tensor(70000.).half().item())
g16 = torch.tensor(1e-8).half().item()
gbf = torch.tensor(1e-8).bfloat16().item()
print(f"1e-8 → float16: {g16}, bfloat16: {gbf:.3e}")
mot_bf = (torch.tensor(1.0, dtype=torch.bfloat16) + 0.001).item()
print("1 + 0.001 trong bfloat16 =", mot_bf)
with torch.autocast("cpu", dtype=torch.bfloat16):
    C = A @ B
sai_so = ((C.float() - A @ B).norm() / (A @ B).norm()).item()
print(f"sai số tương đối của phép nhân bfloat16: {sai_so:.2%}")

float16 lớn nhất: 65504.0 | 70000 thành inf
1e-8 → float16: 0.0, bfloat16: 1.001e-08
1 + 0.001 trong bfloat16 = 1.0
sai số tương đối của phép nhân bfloat16: 0.29%


## Bài tập 8: LR range test cho CNN chữ số {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math, torch, torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
d = load_digits()
X = torch.tensor(d.images / 16, dtype=torch.float32).unsqueeze(1); y = torch.tensor(d.target)
torch.manual_seed(0)
model = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(256, 10))
opt = torch.optim.Adam(model.parameters(), lr=1e-5)
N, LR0, LR1 = 150, 1e-5, 1.0
he_so = (LR1 / LR0) ** (1 / N)
# TODO: lặp N bước, mỗi bước một batch ngẫu nhiên 32 ảnh; ghi lrs, losses (làm mượt trung bình trượt hệ số 0.9); lr_chon

In [ ]:
lrs, losses = [], []
lr, s = LR0, None
for step in range(N):
    for g in opt.param_groups:
        g["lr"] = lr
    i = torch.randint(0, len(X), (32,))
    opt.zero_grad(); loss = nn.functional.cross_entropy(model(X[i]), y[i]); loss.backward(); opt.step()
    s = loss.item() if s is None else 0.9 * s + 0.1 * loss.item()
    lrs.append(lr); losses.append(s)
    if s > 4 * min(losses):
        break
    lr *= he_so
lr_min = lrs[losses.index(min(losses))]
lr_chon = lr_min / 10
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(lrs, losses); ax.set_xscale("log"); ax.axvline(lr_chon, color="gray", ls="--")
ax.set(xlabel="learning rate (thang log)", ylabel="loss (làm mượt)", title=f"Chọn lr ≈ {lr_chon:.0e}")
print(f"loss thấp nhất ở lr ≈ {lr_min:.1e} → chọn lr ≈ {lr_chon:.1e}")

loss thấp nhất ở lr ≈ 2.2e-01 → chọn lr ≈ 2.2e-02
